In [3]:
import torch
import io
import tarfile
import json
import pandas as pd

tab = pd.read_parquet('GlobalGeoTree-6M/GlobalGeoTreeNoMissing.parquet') # tabular data, match sample_id

tar_path = "GlobalGeoTree-6M/GlobalGeoTree-6M/GGT_3500001_3600000-000000.tar"

def process_tarImg(path):
    data = {'ID':[], 'imgData':[]}
    with tarfile.open(path, "r") as tar:
        for member in tar:
            if member.isfile() and member.name.endswith(".pth"):
                # Extract file as bytes
                extracted = tar.extractfile(member)
                
                # Read the file into a BytesIO buffer (makes it seekable)
                buffer = io.BytesIO(extracted.read())
                
                # Load the PyTorch object
                #data = torch.load(buffer, map_location="cpu")

                data['ID'].append(member.name.split('.')[0])
                data['imgData'].append(torch.load(buffer, map_location="cpu"))
                
                #print(member.name, type(data))
    df = pd.DataFrame(data)
    return df
        
df1Tar = process_tarImg(tar_path)
df1Tar.head()



,ID,imgData
0,3500002,"[[[tensor([1648., 1648., 2952., 5172., 3779.],..."
1,3500003,"[[[tensor([218., 284., 328., 207., 207.], dtyp..."
2,3500004,"[[[tensor([3808., 1576., 1576., 882., 709.],..."
3,3500005,"[[[tensor([1750., 1124., 1514., 1976., 1976.],..."
4,3500006,"[[[tensor([1826., 1284., 766., 1110., 1460.],..."


In [ ]:
# Simple CNN for Sentinel image tensors in `df1Tar`
# This cell prepares the image dataset and attempts to merge labels from `tab` (tabular parquet)
# and extract the central pixel from each 5x5 patch, flattening into 120 features per sample.
# Expected: `tab` has columns `sample_id` and `level1_family`.

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Verify df1Tar exists (created by the previous cell)
if 'df1Tar' not in globals():
    raise RuntimeError('`df1Tar` not found in the notebook namespace. Run the tar-processing cell first.')

print(f"df1Tar rows: {len(df1Tar)}; columns: {list(df1Tar.columns)}")

# If tabular metadata `tab` with labels is available, merge on ID.
can_train = 'tab' in globals()
if can_train:
    # Expect `tab` to have a column called `sample_id` matching `ID` in df1Tar
    if 'sample_id' not in tab.columns:
        print("`tab` found but does not contain `sample_id`. Can't merge labels automatically.")
        can_train = False
    elif 'level1_family' not in tab.columns:
        print("`tab` found but does not contain `level1_family` target column. Can't train without labels.")
        can_train = False

# Normalize ID types to string for safe merging
df1Tar['ID'] = df1Tar['ID'].astype(str)
if can_train:
    tab['sample_id'] = tab['sample_id'].astype(int).astype(str)

if can_train:
    print('Merging `df1Tar` with tabular `tab` on ID/sample_id...')
    meta = tab[['sample_id', 'level1_family']].rename(columns={'sample_id':'ID'})
    merged = df1Tar.merge(meta, on='ID', how='inner')
    print(f'Merged samples: {len(merged)} (out of {len(df1Tar)} images)')
    if len(merged) == 0:
        raise RuntimeError('No matching IDs between `df1Tar` and `tab`. Check ID / sample_id values or leading zeros.')

    # --- Extract central pixel from each 5x5 patch and flatten to 120 features ---
    print('Extracting center pixel (middle of 5x5) from each image and flattening to 120 features...')

    def extract_center_flat(img):
        # img: torch.Tensor or array with shape (12, 10, 5, 5)
        if not isinstance(img, torch.Tensor):
            t = torch.tensor(img)
        else:
            t = img
        t = t.detach().cpu()
        if t.ndim != 4:
            raise ValueError(f'Unexpected imgData ndim={t.ndim}, expected 4')
        h = t.shape[2]
        w = t.shape[3]
        cy = h // 2
        cx = w // 2
        center = t[:, :, cy, cx]  # shape (12, 10)
        return center.numpy().ravel()  # length 120

    # Vectorized extraction (Python loop but OK for merged subset)
    feat_list = [extract_center_flat(im) for im in merged['imgData']]
    feats_arr = np.stack(feat_list, axis=0)  # shape (n_samples, 120)
    n_samples, n_feats = feats_arr.shape
    print(f'Extracted features array shape: {feats_arr.shape}')

    # Create column names and attach to merged dataframe
    feat_names = [f'img_center_{i:03d}' for i in range(n_feats)]
    feats_df = pd.DataFrame(feats_arr, columns=feat_names)
    feats_df['ID'] = merged['ID'].values

    # Attach features to merged dataframe
    merged = pd.concat([merged.reset_index(drop=True), feats_df.drop(columns=['ID']).reset_index(drop=True)], axis=1)
    print(f'Merged dataframe now has columns: {len(merged.columns)} (including {n_feats} image features)')

    # Merge features back into the main tab dataframe (left join to preserve tab rows)
    print('Merging image center features back into `tab` as `tab_with_img_features`...')
    tab_with_img_features = tab.merge(feats_df, left_on='sample_id', right_on='ID', how='left')
    num_with_features = tab_with_img_features[feat_names[0]].notna().sum()
    print(f'Number of tab rows with image features: {num_with_features} (out of {len(tab)})')

    # Replace tab variable if desired (commented out); currently we keep `tab_with_img_features`
    # tab = tab_with_img_features

else:
    print('\n`tab` with labels not available or missing columns. The cell will prepare a demo dataset (images only).')
    # Still extract center pixels for df1Tar so you can inspect or attach later
    print('Extracting center pixels for df1Tar (demo run)...')
    def extract_center_flat(img):
        if not isinstance(img, torch.Tensor):
            t = torch.tensor(img)
        else:
            t = img
        t = t.detach().cpu()
        if t.ndim != 4:
            raise ValueError(f'Unexpected imgData ndim={t.ndim}, expected 4')
        h = t.shape[2]
        w = t.shape[3]
        cy = h // 2
        cx = w // 2
        center = t[:, :, cy, cx]
        return center.numpy().ravel()

    feat_list = [extract_center_flat(im) for im in df1Tar['imgData']]
    feats_arr = np.stack(feat_list, axis=0)
    n_samples, n_feats = feats_arr.shape
    print(f'Extracted features array shape (demo): {feats_arr.shape}')
    feat_names = [f'img_center_{i:03d}' for i in range(n_feats)]
    feats_df = pd.DataFrame(feats_arr, columns=feat_names)
    feats_df['ID'] = df1Tar['ID'].values
    df1Tar_with_features = pd.concat([df1Tar.reset_index(drop=True), feats_df.drop(columns=['ID']).reset_index(drop=True)], axis=1)
    print('Created `df1Tar_with_features` (demo). To attach to `tab`, provide `tab` and re-run this cell.')


df1Tar rows: 77552; columns: ['ID', 'imgData']
Merging `df1Tar` with tabular `tab` on ID/sample_id...
Merging `df1Tar` with tabular `tab` on ID/sample_id...
Merged samples: 70733 (out of 77552 images)
Merged samples: 70733 (out of 77552 images)
